AWL Human-in-the-Loop Workflow Demo - [Open this notebook in JupyterLite](https://repolab.github.io/jupyterlite-playground/lab/index.html?fromURL=https://raw.githubusercontent.com/OO-LD/awl-python/refs/heads/main/examples/human_in_the_loop_async.ipynb)

Install required packages. In a pyodide/jupyterlite environment this may take ~30 seconds

In [ ]:
import sys
if sys.platform == 'emscripten':
    # see: https://github.com/pyodide/pyodide/issues/5834
    import micropip
    micropip.uninstall(['typing-extensions'])
%pip install 'bokeh==3.7.3' 'jupyter_bokeh' 'panel==1.7.5' 'awl[hitl]'

Define data models and a workflow. Functions / steps decorated with hitl (human in the loop) will prompt the user for input by an auto-generated UI (see also [OO-LD UI Demo](https://github.com/OO-LD/oold-python/blob/main/examples/linked_data_editor.ipynb)).

*Note: async workflow only required for pyodide/jupyterlite. In a regular jupyter or python environment you can also use sync code, see [here](https://github.com/OO-LD/awl-python/blob/main/examples/human_in_the_loop.ipynb)* 

In [ ]:
from enum import Enum
import json
from typing import Any, Optional
from oold.model import LinkedBaseModel
from pydantic import Field
from awl.hitl import entry_point, hitl

class MachineParams(LinkedBaseModel):
    """Parameters transferred to the machine"""
    model_config = {
        "json_schema_extra": {
            "required": ["param1"],
        },
    }
    param1: int = Field(50, ge=0, le=100)

class Quality(str, Enum):
    good = "Good"
    bad = "Bad"
    worse = "Worse"

class ProcessDocumentation(LinkedBaseModel):
    """Visual result inspection"""
    quality: Quality
    """Good is defined as..."""

class YesNoEnum(Enum):
    yes = True
    no = False

class YesNo(LinkedBaseModel):
    answer: bool # YesNoEnum

@hitl
async def set_machine_params(params: MachineParams):
    # transfer params to machine
    return params

@hitl
async def document_result(params: ProcessDocumentation):
    # validate documentation
    return params

@hitl
async def document_more(params: YesNo) -> YesNo:
    return params

def archive_data(params: Any):
    # store documentation in database
    pass

@entry_point(gui=True, jupyter=True)
async def workflow():
    print("Enter")
    machine_params = await set_machine_params() # prompts user
    do_document_more = True
    while(do_document_more):
        result_evaluation = await document_result() # prompts user
        do_document_more = (await document_more()).answer
        
    print("Machine parameters: ", machine_params)
    print("Result evaluation: ", result_evaluation)
    archive_data(machine_params) # runs automatically
    archive_data(result_evaluation)  # runs automatically
    
await workflow()
print("Done")